In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor

du_lieu = pd.read_csv("../data/clean/clean_1.csv")
bang_train = pd.read_csv("../data/clean/train.csv")
bang_test = pd.read_csv("../data/clean/test.csv")

# Chia vòng khoảng cách đến chợ Bến Thành
vong = [0, 5, 10, 15, 20, 30]
nhan_vong = ["0–5 km", "5–10 km", "10–15 km", "15–20 km", "20–30 km"]
du_lieu["Vong"] = pd.cut(du_lieu["Dist_BenThanh_km"], vong, labels=nhan_vong)

In [2]:
# Số liệu thô: so chung tất cả
tho = du_lieu.groupby("Position").agg(So_can=("Price_per_m2", "size"),
                                       Gia_m2_trung_vi=("Price_per_m2", "median"),
                                       Cach_trung_tam_km=("Dist_BenThanh_km", "median")).round(1)
display(tho)

# So trong cùng vòng khoảng cách
hai_loai = du_lieu[du_lieu["Position"].isin(["Trong hẻm", "Đường chính"])]
theo_vong = hai_loai.pivot_table(index="Vong", columns="Position", values="Price_per_m2",
                                 aggfunc="median", observed=True).round(1)
theo_vong.to_csv("../report/dao_sau_hem_theo_vong.csv", encoding="utf-8-sig")
theo_vong

,So_can,Gia_m2_trung_vi,Cach_trung_tam_km
Position,,,
Không rõ,1343,88.4,11.3
Trong hẻm,1666,94.2,9.6
Đường chính,1481,92.2,10.8


Position,Trong hẻm,Đường chính
Vong,,
0–5 km,165.6,191.9
5–10 km,107.1,138.2
10–15 km,81.2,85.9
15–20 km,39.0,44.4
20–30 km,34.4,35.4


In [3]:
# Các phường/khu thuộc Bình Dương cũ nằm trong bán kính 30 km (nhận diện gần đúng theo tên)
tu_khoa_bd = ["Thủ Dầu Một", "Dĩ An", "Thuận An", "Thuận Giao", "Phú Lợi", "Lái Thiêu", "Bình Dương",
              "Tân Đông Hiệp", "Đông Hòa", "Bình Hòa", "An Phú,", "Chánh Hiệp", "Phú Mỹ,", "Tân Uyên", "Bến Cát"]
la_bd = bang_train["Location"].str.contains("|".join(tu_khoa_bd))
print("Số căn Bình Dương cũ trong tập train:", la_bd.sum(), f"({la_bd.mean():.1%})")

cot = ["Area", "Width", "Length", "Bedrooms", "Floors", "Dist_BenThanh_km", "Dist_Metro_km"]
def chay_ols(bang):
    x = bang[cot].copy()
    for loai in ["Trong hẻm", "Đường chính"]:
        x["Position_" + loai] = (bang["Position"] == loai).astype(int)
    return sm.OLS(bang["Log_Price_per_m2"], sm.add_constant(x)).fit(cov_type="HC3")

ket_qua = []
for ten, bang in [("Toàn bộ dữ liệu", bang_train), ("Bỏ Bình Dương cũ", bang_train[~la_bd])]:
    m = chay_ols(bang)
    dong = {"Du_lieu": ten, "R2": round(m.rsquared, 3)}
    for bien in ["Dist_BenThanh_km", "Floors", "Position_Trong hẻm", "Dist_Metro_km"]:
        dong[bien + "_%"] = round((np.exp(m.params[bien]) - 1) * 100, 1)
        dong[bien + "_p"] = round(m.pvalues[bien], 3)
    ket_qua.append(dong)
bang_bd = pd.DataFrame(ket_qua)
bang_bd.to_csv("../report/dao_sau_bo_binh_duong.csv", index=False, encoding="utf-8-sig")
bang_bd

Số căn Bình Dương cũ trong tập train: 696 (19.4%)


,Du_lieu,R2,Dist_BenThanh_km_%,Dist_BenThanh_km_p,Floors_%,Floors_p,Position_Trong hẻm_%,Position_Trong hẻm_p,Dist_Metro_km_%,Dist_Metro_km_p
0,Toàn bộ dữ liệu,0.678,-5.9,0.0,13.4,0.0,-20.1,0.0,-0.5,0.067
1,Bỏ Bình Dương cũ,0.634,-7.3,0.0,13.1,0.0,-20.0,0.0,-0.7,0.015


In [4]:
dac_trung = ["Area", "Width", "Length", "Floors", "Bedrooms", "Position", "Dist_BenThanh_km", "Dist_Metro_km"]
X_train = pd.get_dummies(bang_train[dac_trung], columns=["Position"], drop_first=True)
X_test = pd.get_dummies(bang_test[dac_trung], columns=["Position"], drop_first=True).reindex(columns=X_train.columns, fill_value=0)

rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=5,
                           min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train, bang_train["Log_Price_per_m2"])

kiem_tra = bang_test.copy()
kiem_tra["Du_doan_gia_m2"] = np.exp(rf.predict(X_test))
kiem_tra["Lech_%"] = (kiem_tra["Du_doan_gia_m2"] / kiem_tra["Price_per_m2"] - 1) * 100   # dương: đoán cao hơn thực tế
kiem_tra["Muc_gia"] = pd.qcut(kiem_tra["Price_per_m2"], 4, labels=["Rẻ nhất 25%", "Thấp–vừa", "Vừa–cao", "Đắt nhất 25%"])
kiem_tra["Vong"] = pd.cut(kiem_tra["Dist_BenThanh_km"], vong, labels=nhan_vong)

theo_muc_gia = kiem_tra.groupby("Muc_gia", observed=True)["Lech_%"].agg(So_can="size", Lech_trung_vi="median").round(1)
theo_vong_kc = kiem_tra.groupby("Vong", observed=True)["Lech_%"].agg(So_can="size", Sai_tuyet_doi_trung_vi=lambda s: s.abs().median()).round(1)
theo_muc_gia.to_csv("../report/dao_sau_sai_so_rf.csv", encoding="utf-8-sig")
display(theo_muc_gia)
theo_vong_kc

,So_can,Lech_trung_vi
Muc_gia,,
Rẻ nhất 25%,225,11.7
Thấp–vừa,225,6.1
Vừa–cao,223,-2.9
Đắt nhất 25%,225,-19.9


,So_can,Sai_tuyet_doi_trung_vi
Vong,,
0–5 km,53,25.6
5–10 km,364,15.9
10–15 km,256,14.8
15–20 km,53,20.9
20–30 km,172,14.5
